# Autonomous IT Tech Support Agent with LangGraph & Multi-Tool RAG
### Stateful Agentic AI Support Pipeline Built with LangGraph & Google Gemini

---

## Project Introduction
In this project, I built an autonomous IT tech support agent using **LangGraph**, **LangChain**, and **Google Gemini**. I designed it to help employees troubleshoot corporate VPN issues by first retrieving answers from an internal PDF knowledge base (RAG), and—when documentation alone cannot resolve the problem—executing deterministic actions against a local SQLite database to create support tickets, check VPN account status, reactivate dormant accounts, provision new users, and update ticket records across multi-turn conversations.

---

## Architectural Workflow
Here is the stateful reasoning and tool-routing loop I implemented in LangGraph:
```
             Employee Inquiry
                    │
                    ▼
         ┌─────────────────────┐
         │ LangGraph Chatbot   │
         │ (State & System LLM)│
         └─────────────────────┘
                    │
       Tool Calls   │   Direct Response
         ┌──────────┴──────────┐
         ▼                     ▼
┌──────────────────┐  ┌──────────────────┐
│   ToolNode       │  │ User Response    │
│ (Dynamic Router) │  └──────────────────┘
└──────────────────┘
         │
         ├──► [1] Knowledge Base Retrieval (FAISS Vector Store RAG)
         ├──► [2] Create Support Ticket (SQLite Tickets DB)
         ├──► [3] Check / Reactivate User (SQLite User Access DB)
         ├──► [4] Provision New VPN User (Account Provisioning)
         └──► [5] Update Support Ticket Status (Lifecycle Tracking)
         │
         ▼
┌──────────────────┐
│ Return to State  │ ──► Iterative Reasoning Loop
└──────────────────┘
```

---

## Tech Stack I Used
- **Orchestration:** LangGraph (`StateGraph`, `ToolNode`, `tools_condition`, `MessagesState`)
- **Framework:** LangChain Core, LangChain Community, LangChain Google GenAI
- **Knowledge Retrieval (RAG):** FAISS vector store, Google Gemini embeddings (`models/gemini-embedding-001`), `PyPDFLoader`, `RecursiveCharacterTextSplitter`
- **Database Layer:** SQLite3 (`support_demo.db`) for persistent support tickets and VPN user access
- **LLM:** Google Gemini via `ChatGoogleGenerativeAI` (configured with `GEMINI_MODEL`, defaulting to `gemini-1.5-flash`)
- **Notebook Rendering:** `IPython.display` (`Markdown`, `display`) for clean multi-turn responses

---

## How I Structured This Notebook
1. **Module 1:** Setting up environment variables and initializing the SQLite database with sample employee records.
2. **Module 2:** Parsing `vpn_user_guide.pdf`, indexing chunks into FAISS with Gemini embeddings, and registering the retriever tool.
3. **Module 3:** Implementing four SQLite action tools for ticketing, account checks/reactivation, and user provisioning.
4. **Module 4:** Binding all tools to Google Gemini, writing the support workflow system prompt, and compiling the LangGraph state machine.
5. **Module 5:** Testing the agent end-to-end on knowledge base queries and multi-turn escalation flows.

---



## Module 1: Environment Setup & Database Initialization
> **What I did here:** I loaded my environment variables from `.env` and initialized `support_demo.db` (SQLite) with persistent `tickets` and `vpn_user_access` tables, seeded with sample employee records.


In [1]:
import os
import sqlite3
import random
import string
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

DB_PATH = 'support_demo.db'

def init_database(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS tickets (
        ticket_id TEXT PRIMARY KEY,
        employee_name TEXT,
        issue_type TEXT,
        description TEXT,
        reported_time TEXT,
        status TEXT,
        updated_at TEXT
    )
    ''')
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS vpn_user_access (
        employee_name TEXT PRIMARY KEY,
        status TEXT
    )
    ''')
    cursor.execute('SELECT count(*) FROM vpn_user_access')
    if cursor.fetchone()[0] == 0:
        cursor.executemany(
            'INSERT INTO vpn_user_access (employee_name, status) VALUES (?, ?)',
            [('Alice Smith', 'active'), ('Bob Jones', 'inactive'), ('Charlie Brown', 'active')]
        )
    conn.commit()
    conn.close()
    print('Database initialized successfully with test employee records.')

init_database(DB_PATH)


Database initialized successfully with test employee records.


## Module 2: Knowledge Base Document Parsing & FAISS Vector Indexing
> **What I did here:** I parsed `vpn_user_guide.pdf`, split it into overlapping chunks, indexed the embeddings in a local FAISS vector store using Google Gemini (`models/gemini-embedding-001`), and registered it as a retriever tool.


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools.retriever import create_retriever_tool

# Normalize Google API Key
if 'GEMINI_API_KEY' in os.environ and 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GOOGLE_API_KEY'] = os.environ['GEMINI_API_KEY']

embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')

if not os.path.exists('faiss_index'):
    print('Building FAISS knowledge base index from vpn_user_guide.pdf using Gemini embeddings...')
    loader = PyPDFLoader('vpn_user_guide.pdf')
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local('faiss_index')
    print(f'FAISS index created from {len(documents)} pages, {len(chunks)} chunks.')
else:
    vectorstore = FAISS.load_local('faiss_index', embeddings, allow_dangerous_deserialization=True)
    print('Loaded existing FAISS knowledge base index from disk.')

retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
kb_tool = create_retriever_tool(
    retriever,
    name='search_vpn_knowledge_base',
    description='Search Acme Corp internal VPN user guide for installation steps, troubleshooting, and configuration details.'
)


C:\Users\Hp\AppData\Local\Temp\ipykernel_11908\646098276.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Loaded existing FAISS knowledge base index from disk.


## Module 3: Support Action Tools Implementation
> **What I did here:** I implemented deterministic SQLite `@tool` functions for creating support tickets, checking and reactivating dormant VPN accounts, updating ticket statuses, and provisioning new VPN users.


In [3]:
from langchain_core.tools import tool

@tool
def create_support_ticket(employee_name: str, issue_type: str, description: str) -> str:
    '''Create a new support ticket when the knowledge base cannot resolve the employee issue.'''
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    ticket_id = 'T-' + ''.join(random.choices(string.digits, k=6))
    reported_time = datetime.now(timezone.utc).isoformat()
    cursor.execute(
        'INSERT INTO tickets (ticket_id, employee_name, issue_type, description, reported_time, status) VALUES (?, ?, ?, ?, ?, ?)',
        (ticket_id, employee_name, issue_type, description, reported_time, 'open'),
    )
    conn.commit()
    conn.close()
    return f'Support ticket {ticket_id} has been created for {employee_name}. Issue type: {issue_type}. Status: open.'

@tool
def check_vpn_user(employee_name: str) -> str:
    '''Check whether an employee currently has VPN access, and automatically reactivate them if inactive.'''
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT status FROM vpn_user_access WHERE employee_name = ?', (employee_name,))
    row = cursor.fetchone()
    if row is None:
        conn.close()
        return f'employee_name={employee_name}, status=inactive, was_reactivated=False, vpn_access=False (employee not found in VPN system)'
    status = row[0]
    was_reactivated = False
    if status == 'inactive':
        cursor.execute('UPDATE vpn_user_access SET status = ? WHERE employee_name = ?', ('active', employee_name))
        conn.commit()
        status = 'active'
        was_reactivated = True
    conn.close()
    return f'employee_name={employee_name}, status={status}, was_reactivated={was_reactivated}, vpn_access={status == "active"}'

@tool
def update_ticket(ticket_id: str, status: str, description: str) -> str:
    '''Update an existing support ticket status and append a note to its description.'''
    valid_statuses = ['open', 'in_progress', 'resolved', 'closed', 'pending']
    if status.lower() not in valid_statuses:
        return f'Invalid status "{status}". Must be one of: {", ".join(valid_statuses)}.'
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT description FROM tickets WHERE ticket_id = ?', (ticket_id,))
    row = cursor.fetchone()
    if row is None:
        conn.close()
        return f'No ticket found with ID {ticket_id}.'
    existing_description = row[0] or ''
    updated_at = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    appended_description = f'{existing_description}\n\n[UPDATE {updated_at}]: {description}'
    cursor.execute('UPDATE tickets SET status = ?, description = ?, updated_at = ? WHERE ticket_id = ?', (status.lower(), appended_description, updated_at, ticket_id))
    conn.commit()
    conn.close()
    return f'Ticket {ticket_id} updated - status: {status.lower()}, note added.'

@tool
def provision_vpn_user(employee_name: str) -> str:
    '''Provision VPN access for an employee who was not found in the VPN system.'''
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT status FROM vpn_user_access WHERE employee_name = ?', (employee_name,))
    row = cursor.fetchone()
    if row is not None:
        cursor.execute('UPDATE vpn_user_access SET status = ? WHERE employee_name = ?', ('active', employee_name))
        conn.commit()
        conn.close()
        return f'employee_name={employee_name}, status=active, user_already_existed=True'
    cursor.execute('INSERT INTO vpn_user_access (employee_name, status) VALUES (?, ?)', (employee_name, 'active'))
    conn.commit()
    conn.close()
    return f'employee_name={employee_name}, status=active, user_already_existed=False'


## Module 4: LangGraph State Machine Compilation & Conditional Routing
> **What I did here:** I bound the RAG and SQLite tools to Google Gemini (`ChatGoogleGenerativeAI`), wrote the step-by-step IT support system prompt, and compiled the cyclic LangGraph state graph (`chatbot` ↔ `tools`).


In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage

gemini_model = os.environ.get('GEMINI_MODEL', 'gemini-1.5-flash')
llm = ChatGoogleGenerativeAI(model=gemini_model, temperature=0)
tools = [kb_tool, create_support_ticket, check_vpn_user, update_ticket, provision_vpn_user]
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = '''You are a helpful IT support assistant for Acme Corp. You assist employees with VPN-related issues.

Follow this process in order:
1. Always search the VPN knowledge base first for setup, troubleshooting, or configuration questions.
2. If the knowledge base cannot resolve the issue, collect the employee name, issue type, and description, then create a support ticket. Never create ticket without asking the employee name first.
3. Immediately after creating a ticket, ask the employee if they would like you to check their VPN account status to help resolve things faster.
4. If they agree, call check_vpn_user with their employee name:
   - status=active, was_reactivated=False: tell them access is active and ask to retry.
   - status=active, was_reactivated=True: tell them account was inactive and has been reactivated.
   - status=inactive, was_reactivated=False: ask if they want a new VPN account created.
5. If they agree to account creation, call provision_vpn_user with their name.
6. After telling them to try again, wait for their response:
   - If fixed, call update_ticket to close the ticket with a note.
   - If still not working, call update_ticket to keep ticket open with a note.
7. If they decline the status check, leave the ticket open for IT.

Whenever a tool returns a ticket ID or status, repeat that exact value back to the user.
Always be polite, concise, and stay focused on VPN-related topics only.'''

def chatbot(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages': [response]}

graph_builder = StateGraph(MessagesState)
graph_builder.add_node('chatbot', chatbot)
graph_builder.add_node('tools', ToolNode(tools=tools))
graph_builder.add_edge(START, 'chatbot')
graph_builder.add_conditional_edges('chatbot', tools_condition)
graph_builder.add_edge('tools', 'chatbot')

agent = graph_builder.compile()
print('LangGraph state machine successfully compiled with Google Gemini.')


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


LangGraph state machine successfully compiled with Google Gemini.


## Module 5: End-to-End Simulation & Verification Tests
> **What I did here:** I verified my compiled agent end-to-end across single-turn knowledge base retrieval and multi-turn VPN issue escalation scenarios, rendering the outputs as formatted Markdown.


In [8]:
from IPython.display import Markdown, display

# Test Scenario 1: Knowledge Base Query
print('--- Test 1: Knowledge Base Query ---')
query_1 = {'messages': [{'role': 'user', 'content': 'What are the system requirements and setup steps for connecting to the Acme Corp VPN on Windows?'}]}
res_1 = agent.invoke(query_1)
msg_1 = res_1['messages'][-1]
display(Markdown(f'**Agent Response:**\n\n{msg_1.text or msg_1.content}'))


--- Test 1: Knowledge Base Query ---


C:\Users\Hp\AppData\Roaming\Python\Python314\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Hp\AppData\Roaming\Python\Python314\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Agent Response:**

Here are the system requirements and setup steps for connecting to the Acme Corp VPN on Windows based on the user guide:

### **System Requirements**
* **Software:** Forcepoint VPN Client version 6.8 or higher for Windows.

### **Setup Steps (Installation)**
1. Double-click the installation executable file to open the **Forcepoint VPN Client Setup** window.
2. Click **Install**.
3. Click **Next** and follow the on-screen prompts.
   * *Note:* Allow the installation of all drivers and components if prompted by confirmation requests.
4. Once installation is complete, connect to a new gateway to start using the Forcepoint VPN Client.

In [9]:
from IPython.display import Markdown, display

# Test Scenario 2: Issue Escalation, Ticket Creation & Account Reactivation
print('--- Test 2: Ticket Creation & User Verification ---')
conversation = [
    {'role': 'user', 'content': 'I am getting error 789 connection failed. My name is Bob Jones.'},
]
res_2 = agent.invoke({'messages': conversation})
msg_2 = res_2['messages'][-1]
display(Markdown(f'**Turn 1 Response:**\n\n{msg_2.text or msg_2.content}'))

# User agrees to status check
conversation = res_2['messages'] + [{'role': 'user', 'content': 'Yes, please check my VPN account status.'}]
res_3 = agent.invoke({'messages': conversation})
msg_3 = res_3['messages'][-1]
display(Markdown(f'**Turn 2 Response:**\n\n{msg_3.text or msg_3.content}'))


--- Test 2: Ticket Creation & User Verification ---


C:\Users\Hp\AppData\Roaming\Python\Python314\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Hp\AppData\Roaming\Python\Python314\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Hp\AppData\Roaming\Python\Python314\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Turn 1 Response:**

Support ticket T-739852 has been created for Bob Jones. Issue type: Connection Error 789. Status: open.

Would you like me to check your VPN account status to help resolve things faster?

C:\Users\Hp\AppData\Roaming\Python\Python314\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Hp\AppData\Roaming\Python\Python314\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


**Turn 2 Response:**

Your VPN access is currently active. Please try connecting again and let me know if it works.